In [9]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!ls -lh "/content/drive/MyDrive/food.zip"

-rw------- 1 root root 1.5G Aug 12 08:11 /content/drive/MyDrive/food.zip


In [11]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  19% /


In [12]:
import os

zip_path = "/content/drive/MyDrive/food.zip"
extract_path = "/content"

os.makedirs(extract_path, exist_ok=True)

!unzip -q "$zip_path" -d "$extract_path"

print("Extraction completed!")

error:  zipfile read error
error [/content/drive/MyDrive/food.zip]:  start of central directory not found;
  zipfile corrupt.
  (please check that you have transferred or created the zipfile in the
  appropriate BINARY mode and that you have compiled UnZip properly)
Extraction completed!


In [13]:
!find "/content/food" -maxdepth 2 -type d | head -100

find: ‘/content/food’: No such file or directory


In [8]:
import os
os.listdir("/content/food/")

FileNotFoundError: [Errno 2] No such file or directory: '/content/food/'

In [ ]:
os.listdir("/content/food/train/")

In [ ]:
os.listdir("/content/food/test/")

In [ ]:
os.listdir("/content/food/valid/")

In [ ]:
os.listdir("/content/food/train/Donut/")

In [ ]:
os.listdir("/content/food/test/Donut/")

In [ ]:
os.listdir("/content/food/valid/Donut/")

In [ ]:
train_data_path="/content/food/train"
valid_data_path="/content/food/valid"
test_data_path="/content/food/test"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(5,5))
plt.title("Baked Potato")
baked_potato=plt.imread(train_data_path+"/Baked Potato/Baked Potato-Train (100).jpeg")
plt.imshow(baked_potato)
plt.show()

In [ ]:
plt.figure(figsize=(5,5))
plt.title("Apple")
apple_pie=plt.imread(train_data_path+"/apple_pie/1043283.jpg")
plt.imshow(apple_pie)
plt.show()

In [ ]:
label_names=['Baked Potato','Crispy Chicken','Donut','Fries','Hot Dog','Sandwich','Taco','Taquito','apple_pie','burger','butter_naan','chai','chapati','cheesecake','chicken_curry','chole_bhature','dal_makhani','dhokla','fried_rice','ice_cream','idli','jalebi','kaathi_rolls','kadai_paneer','kulfi','masala_dosa','momos','omelette','paani_puri','pakode','pav_bhaji','pizza','samosa','sushi']

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_data_rules = ImageDataGenerator(rescale = 1 / 255.0 ,
                                      rotation_range = 0.2 ,
                                      shear_range=0.2 ,
                                      horizontal_flip = True)

val_data_rules = ImageDataGenerator(rescale = 1 / 255.0 ,
                                      rotation_range = 0.2 ,
                                      shear_range=0.2 ,
                                      horizontal_flip = True)

In [ ]:
final_train_data = train_data_rules.flow_from_directory(train_data_path , target_size=(256,256) ,
                                     color_mode='rgb',
                                     classes=label_names , class_mode='categorical')

final_val_data = val_data_rules.flow_from_directory(valid_data_path , target_size=(256,256) ,
                                     color_mode='rgb',
                                     classes=label_names , class_mode='categorical')

VGG16 Architecture

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import Flatten,Dense
from tensorflow.keras.models import Model
from tensorflow.keras.activations import sigmoid,relu


vgg16_model = VGG16(input_shape=[256,256,3], weights='imagenet' , include_top=False)

for i in vgg16_model.layers:
  i.trainable = False

inputs_to_ann = Flatten()(vgg16_model.output)

# give Flatten values to ANN
h1_out = Dense(units=128,kernel_initializer='he_uniform',activation='relu')(inputs_to_ann)
h2_out = Dense(units=64,kernel_initializer='he_uniform',activation='relu')(h1_out)
h3_out = Dense(units=38,kernel_initializer='he_uniform',activation='relu')(h2_out)
output = Dense(units=34,kernel_initializer='glorot_uniform',activation='softmax')(h3_out)

model = Model(inputs = vgg16_model.input , outputs = output)

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['Accuracy'])

In [ ]:
model.fit(final_train_data,
          validation_data=final_val_data,
          batch_size=100,
          epochs=20)

In [ ]:
import cv2

In [ ]:
def testing_function(image_path):
  t_image = cv2.imread(image_path,1)
  print(f"Original_image size : {t_image.shape}")
  resized_image = cv2.resize(t_image , (256,256))
  print(f"Resized Image size : {resized_image.shape}")
  scaled_down_image = resized_image / 255.0
  print(f"After Scaling : {scaled_down_image.shape}")
  added_dim_image = np.expand_dims(scaled_down_image , axis=0)
  print(f"After Adding 1 Dim size : {added_dim_image.shape}")
  if model.predict(added_dim_image)[0][0] > 0.5:
    print(f"Patient Cell is having Malaria")
  else:
    print(f"Patient Cell is Not Having Malaria")

  plt.imshow(t_image[: , : ,::-1])
  plt.show()

In [ ]:
testing_function("/content/Malaria Dataset/test/Parasitized/C101P62ThinF_IMG_20150918_151006_cell_74_png.rf.168fe2682fabe117672de8350b6b1796.jpg")